In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T

# Detection models from torchvision
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# If needed, install extra libs:
# !pip install torchmetrics  # optional


In [ ]:
# Download the 2007 trainval tar file (446 MB)
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar

# Extract it
!tar -xf VOCtrainval_06-Nov-2007.tar

# Check the resulting directory structure
!ls -R VOCdevkit


In [ ]:
# Download the 2007 test tar file (438 MB)
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar

# Extract
!tar -xf VOCtest_06-Nov-2007.tar

# Check that we now also have the test set
!ls -R VOCdevkit


In [ ]:
root_dir = "VOCdevkit"
year = "2007"

trainval_file = f"{root_dir}/VOC{year}/ImageSets/Main/trainval.txt"
test_file = f"{root_dir}/VOC{year}/ImageSets/Main/test.txt"

# Example: read the file with image IDs
with open(trainval_file, 'r') as f:
    lines = f.read().strip().split()
print("Number of trainval IDs:", len(lines))

# Check if the test file exists
import os
print("test.txt exists:", os.path.exists(test_file))


In [ ]:
class VOCDataset(Dataset):
    def __init__(self, root, year="2007", imageset="trainval", transforms=None, class_to_idx=None):
        """
        root: Path to the root folder containing VOCdevkit
        year: e.g., "2007" or "2012"
        imageset: e.g., "trainval", "test"
        transforms: optional torchvision transforms
        class_to_idx: dictionary mapping class_name -> class_id (int)
                      If None, we'll create a default from the usual VOC classes.
        """
        self.root = root
        self.year = year
        self.imageset = imageset
        self.transforms = transforms

        # Default PASCAL VOC classes (20 classes) + background(0)
        if class_to_idx is None:
            self.class_to_idx = {
                "background": 0,
                "aeroplane": 1,
                "bicycle": 2,
                "bird": 3,
                "boat": 4,
                "bottle": 5,
                "bus": 6,
                "car": 7,
                "cat": 8,
                "chair": 9,
                "cow": 10,
                "diningtable": 11,
                "dog": 12,
                "horse": 13,
                "motorbike": 14,
                "person": 15,
                "pottedplant": 16,
                "sheep": 17,
                "sofa": 18,
                "train": 19,
                "tvmonitor": 20
            }
        else:
            self.class_to_idx = class_to_idx

        # Load the image IDs from the text file
        imageset_path = os.path.join(self.root, f"VOC{year}", "ImageSets", "Main", f"{imageset}.txt")
        with open(imageset_path, "r") as f:
            self.image_ids = [line.strip() for line in f.readlines()]

        # Paths
        self.annotations_path = os.path.join(self.root, f"VOC{year}", "Annotations")
        self.images_path = os.path.join(self.root, f"VOC{year}", "JPEGImages")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        # Load image
        img_file = os.path.join(self.images_path, f"{image_id}.jpg")
        img = Image.open(img_file).convert("RGB")

        # Parse XML
        xml_file = os.path.join(self.annotations_path, f"{image_id}.xml")
        tree = ET.parse(xml_file)
        root = tree.getroot()

        boxes = []
        labels = []

        for obj in root.findall("object"):
            name = obj.find("name").text.lower().strip()
            bbox = obj.find("bndbox")
            if bbox is None:
                continue
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(self.class_to_idx[name])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([index])

        if self.transforms is not None:
            img = self.transforms(img)

        return img, target


In [ ]:
train_transforms = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor()
])

val_transforms = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor()
])


In [ ]:


train_dataset = VOCDataset(
    root=root_dir,
    year=2007,
    imageset="trainval",  # or "train"
    transforms=train_transforms
)

val_dataset = VOCDataset(
    root=root_dir,
    year=2007,
    imageset="test",      # or "val" if you have a separate val file
    transforms=val_transforms
)

# DataLoaders
# Note: For object detection, we often need a custom collate_fn to handle variable # of objects
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)


In [ ]:
import torch
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.detection.ssd import SSDClassificationHead

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# The PASCAL VOC has 20 classes (+ 1 for background = 21).
num_classes = 21

# Load the pretrained SSD300 with VGG16 backbone
ssd_model = ssd300_vgg16(weights="DEFAULT").to(device)

# As of newer TorchVision releases, 'ssd300_vgg16' typically has 6 feature map scales
# with the following channel/anchor sizes for classification:
in_channels = [512, 1024, 512, 256, 256, 256]  # The channels for each conv feature map
num_anchors = [4,   6,    6,   6,   4,   4]    # The number of anchors at each scale

# Create a new classification head with your desired number of classes (21 for VOC)
new_class_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes  # 21 = 20 object classes + 1 background
).to(device)

# Replace the old classification head in the model with the newly created one
ssd_model.head.classification_head = new_class_head


In [ ]:
frcnn_model = fasterrcnn_resnet50_fpn(weights="DEFAULT").to(device)

# Update the classifier head for 21 classes
in_features = frcnn_model.roi_heads.box_predictor.cls_score.in_features
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
frcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes).to(device)


In [ ]:
# Let’s train the SSD model:
model = ssd_model  # or frcnn_model
model.train()

optimizer = optim.SGD([p for p in model.parameters() if p.requires_grad],
                      lr=0.001, momentum=0.9, weight_decay=0.0005)

num_epochs = 2  # Increase if you have enough time/GPU

for epoch in range(num_epochs):
    total_loss = 0.0
    for images, targets in train_loader:
        images = list(img.to(device) for img in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)  # model in training mode returns a dict of losses
        losses = sum(loss for loss in loss_dict.values())
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {avg_loss:.4f}")


In [ ]:
model.eval()

# Get a batch from val_loader
images, targets = next(iter(val_loader))
images = list(img.to(device) for img in images)

with torch.no_grad():
    predictions = model(images)

for i, pred in enumerate(predictions):
    print(f"Image {i} predictions:")
    print("Boxes:", pred["boxes"])
    print("Labels:", pred["labels"])
    print("Scores:", pred["scores"])
    print("---")


In [ ]:
def plot_predictions(img_tensor, pred_dict, threshold=0.5):
    """
    img_tensor: shape [C, H, W]
    pred_dict: dict with 'boxes', 'labels', 'scores'
    threshold: score threshold to display
    """
    boxes = pred_dict["boxes"].cpu().numpy()
    labels = pred_dict["labels"].cpu().numpy()
    scores = pred_dict["scores"].cpu().numpy()

    # Convert image tensor to numpy
    img_np = img_tensor.permute(1,2,0).cpu().numpy()

    plt.figure()
    plt.imshow(img_np)
    ax = plt.gca()

    for box, label, score in zip(boxes, labels, scores):
        if score >= threshold:
            x1, y1, x2, y2 = box
            rect = plt.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, linewidth=2
            )
            ax.add_patch(rect)
            ax.text(
                x1, y1, f"Class: {label} | {score:.2f}",
                fontsize=8, backgroundcolor='white'
            )

    plt.axis("off")
    plt.show()


# Show predictions for the first image in the batch
model.eval()
with torch.no_grad():
    predictions = model(images)

plot_predictions(images[0], predictions[0], threshold=0.2)


In [ ]:
class BottleneckBlock(nn.Module):
    expansion = 4

    def __init__(self, in_channels, mid_channels, stride=1):
        super(BottleneckBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, stride=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)

        self.conv2 = nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(mid_channels)

        self.conv3 = nn.Conv2d(mid_channels, mid_channels * self.expansion, kernel_size=1, stride=1, bias=False)
        self.bn3 = nn.BatchNorm2d(mid_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)

        self.downsample = None
        if stride != 1 or in_channels != mid_channels * self.expansion:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, mid_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(mid_channels * self.expansion)
            )

    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out += identity
        out = self.relu(out)

        return out


In [ ]:
class ResNet50Backbone(nn.Module):
    def __init__(self):
        super(ResNet50Backbone, self).__init__()

        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 3, stride=1)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)  # Final feature layer

    def _make_layer(self, mid_channels, blocks, stride):
        layers = []
        layers.append(BottleneckBlock(self.in_channels, mid_channels, stride))
        self.in_channels = mid_channels * BottleneckBlock.expansion

        for _ in range(1, blocks):
            layers.append(BottleneckBlock(self.in_channels, mid_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x1 = self.layer1(x)
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)

        return OrderedDict([
            ("0", x2),
            ("1", x3),
            ("2", x4)
        ])  # SSD requires an OrderedDict

# Create backbone
backbone = ResNet50Backbone()


In [ ]:
from torchvision.models.detection.ssd import SSD, DefaultBoxGenerator
from collections import OrderedDict

num_classes = 21
in_channels = [512, 1024, 2048]  # Feature maps from ResNet backbone
num_anchors = [4, 6, 6]  # Adjust anchors per layer

# Create SSD Model
ssd_model = SSD(
    backbone=backbone,
    num_classes=num_classes,
    anchor_generator=DefaultBoxGenerator([[2, 3]] * len(in_channels)),
    size=(300, 300),
).to(device)

print(ssd_model)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from torchvision import transforms

# Load Dataset
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
])

dataset = VOCDataset(root=root_dir, year="2007", imageset="train", transforms=transform)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=lambda batch: tuple(zip(*batch)))

# Define optimizer
optimizer = optim.Adam(ssd_model.parameters(), lr=0.001, weight_decay=5e-4)

# Training loop
num_epochs = 10
ssd_model.train()

for epoch in range(num_epochs):
    total_loss = 0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = ssd_model(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")


In [ ]:
import torch
import torch.nn as nn

class DepthwiseSeparableConv(nn.Module):
    """Depthwise Separable Convolution: Reduces computation while maintaining efficiency."""
    def __init__(self, in_channels, out_channels, stride=1):
        super(DepthwiseSeparableConv, self).__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, padding=1, groups=in_channels, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pointwise(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x

class InvertedResidualBlock(nn.Module):
    """Inverted Residual Block: Expands, depthwise conv, then projects."""
    def __init__(self, in_channels, out_channels, expansion_factor, stride):
        super(InvertedResidualBlock, self).__init__()
        mid_channels = in_channels * expansion_factor
        self.use_residual = stride == 1 and in_channels == out_channels

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU6(inplace=True),

            nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=stride, padding=1, groups=mid_channels, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU6(inplace=True),

            nn.Conv2d(mid_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        else:
            return self.conv(x)

class MobileNetV2Backbone(nn.Module):
    """MobileNetV2 Backbone that outputs multiple feature maps for SSD."""
    def __init__(self):
        super(MobileNetV2Backbone, self).__init__()

        self.initial_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True)
        )

        self.inverted_residuals = nn.Sequential(
            InvertedResidualBlock(32, 16, expansion_factor=1, stride=1),
            InvertedResidualBlock(16, 24, expansion_factor=6, stride=2),
            InvertedResidualBlock(24, 24, expansion_factor=6, stride=1),  # Feature Map 1
            InvertedResidualBlock(24, 32, expansion_factor=6, stride=2),
            InvertedResidualBlock(32, 32, expansion_factor=6, stride=1),
            InvertedResidualBlock(32, 32, expansion_factor=6, stride=1),  # Feature Map 2
            InvertedResidualBlock(32, 64, expansion_factor=6, stride=2),
            InvertedResidualBlock(64, 64, expansion_factor=6, stride=1),
            InvertedResidualBlock(64, 64, expansion_factor=6, stride=1),
            InvertedResidualBlock(64, 64, expansion_factor=6, stride=1),
            InvertedResidualBlock(64, 96, expansion_factor=6, stride=1),
            InvertedResidualBlock(96, 96, expansion_factor=6, stride=1),
            InvertedResidualBlock(96, 96, expansion_factor=6, stride=1),  # Feature Map 3
            InvertedResidualBlock(96, 160, expansion_factor=6, stride=2),
            InvertedResidualBlock(160, 160, expansion_factor=6, stride=1),
            InvertedResidualBlock(160, 160, expansion_factor=6, stride=1),
            InvertedResidualBlock(160, 320, expansion_factor=6, stride=1),  # Feature Map 4
        )

        self.out_channels = [24, 32, 96]  # Feature maps at selected layers

    def forward(self, x):
        x = self.initial_conv(x)
        feature_maps = []

        for i, layer in enumerate(self.inverted_residuals):
            x = layer(x)
            if i in [2, 5, 12]:  # Selecting feature maps at these layers
                feature_maps.append(x)

        return OrderedDict([
            ("0", feature_maps[0]),
            ("1", feature_maps[1]),
            ("2", feature_maps[2])
        ])  # SSD expects an OrderedDict


# Create MobileNetV2 Backbone
backbone = MobileNetV2Backbone()


In [ ]:
from torchvision.models.detection.ssd import SSD, DefaultBoxGenerator
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define output feature maps for SSD
in_channels = [96, 320, 512]  # MobileNet feature maps
num_anchors = [4, 6, 6]


class MobileNetV2SSD(nn.Module):
    def __init__(self):
        super(MobileNetV2SSD, self).__init__()
        self.backbone = MobileNetV2Backbone()

    def forward(self, x):
        x = self.backbone(x)
        return OrderedDict([("0", x)])

# Create SSD model with MobileNetV2
from torchvision.models.detection.ssd import SSD, DefaultBoxGenerator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create SSD model using MobileNetV2 as the backbone
ssd_model = SSD(
    backbone=MobileNetV2Backbone(),
    num_classes=21,
    anchor_generator=DefaultBoxGenerator([[2, 3]] * 3),
    size=(300, 300),
).to(device)

print(ssd_model)


# EfficientNet

In [ ]:
# Load Dataset
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
])

dataset = VOCDataset(root=root_dir, year="2007", imageset="train", transforms=transform)
train_loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=lambda batch: tuple(zip(*batch)))

# Define optimizer
optimizer = torch.optim.Adam(ssd_model.parameters(), lr=0.001, weight_decay=5e-4)

# Training loop
num_epochs = 10
ssd_model.train()

for epoch in range(num_epochs):
    total_loss = 0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = ssd_model(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")


In [ ]:
import torch

# Define the save path
model_save_path = "/content/ssd_mobilenet_v2.pth"  # Change name for different backbones

# Save the trained model's weights
torch.save(ssd_model.state_dict(), model_save_path)

print(f"Model saved at {model_save_path}")


In [ ]:
# Load the model architecture
ssd_model = SSD(
    backbone=MobileNetV2Backbone(),  # Use the same backbone as training
    num_classes=21,
    anchor_generator=DefaultBoxGenerator([[2, 3]] * 3),
    size=(300, 300),
).to(device)

# Load the saved model weights
ssd_model.load_state_dict(torch.load(model_save_path))

# Set model to evalu


In [ ]:
import matplotlib.pyplot as plt
VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat",
    "bottle", "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

# Function to plot predictions
def plot_predictions(img_tensor, pred_dict, threshold=0.5):
    """
    img_tensor: shape [C, H, W]
    pred_dict: dict with 'boxes', 'labels', 'scores'
    threshold: score threshold to display
    """
    boxes = pred_dict["boxes"].cpu().numpy()
    labels = pred_dict["labels"].cpu().numpy()
    scores = pred_dict["scores"].cpu().numpy()

    # Convert image tensor to numpy
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()

    plt.figure(figsize=(8, 8))
    plt.imshow(img_np)
    ax = plt.gca()

    for box, label, score in zip(boxes, labels, scores):
        if score >= threshold:
            x1, y1, x2, y2 = box
            rect = plt.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                fill=False, edgecolor="red", linewidth=2
            )
            ax.add_patch(rect)
            ax.text(
                x1, y1, f"{VOC_CLASSES[label]}: {score:.2f}",
                fontsize=8, backgroundcolor='white', color="black"
            )

    plt.axis("off")
    plt.show()


# Set model to eval mode
ssd_model.eval()

# Load one batch for inference
images, targets = next(iter(train_loader))  # Get first batch
images = [img.to(device) for img in images]

# Run inference
with torch.no_grad():
    predictions = ssd_model(images)

# Plot predictions for first image in batch
plot_predictions(images[0].cpu(), predictions[0], threshold=0.1)
